In [ ]:
from google.colab import drive
import os

# Connect Colab to Google Drive
drive.mount("/content/drive")

# Main project location
PROJECT_PATH = "/content/drive/MyDrive/Fraud_Investigation_System"

# Create the project folders
folders = [
    "data/raw",
    "data/processed",
    "models",
    "reports",
    "notebooks"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)

print("Project created successfully!")
print("Project location:", PROJECT_PATH)

Mounted at /content/drive
Project created successfully!
Project location: /content/drive/MyDrive/Fraud_Investigation_System


In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd
import os

# Download the fraud dataset through the OpenML API
X, y = fetch_openml(
    data_id=1597,
    as_frame=True,
    return_X_y=True
)

# Combine the input columns and fraud label
df = X.copy()
df["Class"] = y.astype(int)

# Save the original dataset in Google Drive
raw_file_path = os.path.join(
    PROJECT_PATH,
    "data/raw/creditcard.csv"
)

df.to_csv(raw_file_path, index=False)

print("Dataset downloaded successfully!")
print("Dataset shape:", df.shape)
print("Saved at:", raw_file_path)

df.head()

Dataset downloaded successfully!
Dataset shape: (284807, 30)
Saved at: /content/drive/MyDrive/Fraud_Investigation_System/data/raw/creditcard.csv


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
# Basic dataset information
print("DATASET OVERVIEW")
print("-" * 50)

print("Number of transactions:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nClass distribution:")
print(df["Class"].value_counts())

fraud_percentage = df["Class"].mean() * 100
print(f"\nFraud percentage: {fraud_percentage:.4f}%")

DATASET OVERVIEW
--------------------------------------------------
Number of transactions: 284807
Number of columns: 30

Column names:
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Data types:
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

Missing values: 0
Duplicate rows: 9144

Class distribution:
Class
0    284315
1       492

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

# Start Spark inside Google Colab
spark = (
    SparkSession.builder
    .appName("FraudInvestigationSystem")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

# Hide unnecessary Spark messages
spark.sparkContext.setLogLevel("ERROR")

print("Spark started successfully!")
print("Spark version:", spark.version)

Spark started successfully!
Spark version: 4.0.4


In [ ]:
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
from pyspark.sql.functions import current_timestamp, lit
import os

# File locations
RAW_DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data/raw/creditcard.csv"
)

BRONZE_PATH = os.path.join(
    PROJECT_PATH,
    "data/processed/bronze_transactions"
)

# Define the expected schema instead of letting Spark guess
schema_fields = [
    StructField(f"V{i}", DoubleType(), True)
    for i in range(1, 29)
]

schema_fields.extend([
    StructField("Amount", DoubleType(), True),
    StructField("Class", IntegerType(), True)
])

transaction_schema = StructType(schema_fields)

# Read the original CSV using Spark
bronze_df = (
    spark.read
    .option("header", True)
    .schema(transaction_schema)
    .csv(RAW_DATA_PATH)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("creditcard.csv"))
)

# Save the Bronze dataset as Parquet
(
    bronze_df.write
    .mode("overwrite")
    .parquet(BRONZE_PATH)
)

print("Bronze layer created successfully!")
print("Bronze rows:", bronze_df.count())
print("Bronze columns:", len(bronze_df.columns))
print("Saved at:", BRONZE_PATH)

bronze_df.show(5, truncate=False)

Bronze layer created successfully!
Bronze rows: 284807
Bronze columns: 32
Saved at: /content/drive/MyDrive/Fraud_Investigation_System/data/processed/bronze_transactions
+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+------+-----+--------------------------+--------------+
|V1                |V2                 |V3              |V4                |V5                 |V6                 |V7                 |V8                |V9                |V10                |V11               |V12               |V1

In [ ]:
from functools import reduce
from operator import and_

from pyspark.sql.functions import (
    col,
    concat,
    lit,
    lpad,
    monotonically_increasing_id
)

SILVER_PATH = os.path.join(
    PROJECT_PATH,
    "data/processed/silver_transactions"
)

QUARANTINE_PATH = os.path.join(
    PROJECT_PATH,
    "data/processed/quarantined_transactions"
)

bronze_loaded_df = spark.read.parquet(BRONZE_PATH)

source_columns = [
    f"V{i}" for i in range(1, 29)
] + ["Amount", "Class"]

# Required fields must not be null
no_nulls_rule = reduce(
    and_,
    [col(column).isNotNull() for column in source_columns]
)

# Data-quality rules
valid_record_rule = (
    no_nulls_rule
    & (col("Amount") >= 0)
    & (col("Class").isin(0, 1))
)

# Separate invalid records
quarantine_df = bronze_loaded_df.filter(~valid_record_rule)

# Preserve every valid source record
valid_df = bronze_loaded_df.filter(valid_record_rule)

# Create a unique ingestion-level transaction ID
silver_df = valid_df.withColumn(
    "transaction_id",
    concat(
        lit("txn_"),
        lpad(
            monotonically_increasing_id().cast("string"),
            12,
            "0"
        )
    )
)

# Standardise column names
for old_name in silver_df.columns:
    silver_df = silver_df.withColumnRenamed(
        old_name,
        old_name.lower()
    )

# Save corrected Silver and quarantine layers
silver_df.write.mode("overwrite").parquet(SILVER_PATH)
quarantine_df.write.mode("overwrite").parquet(QUARANTINE_PATH)

# Duplicate-looking signatures are measured, not deleted
duplicate_signature_count = (
    valid_df
    .groupBy(source_columns)
    .count()
    .filter(col("count") > 1)
    .selectExpr("SUM(count - 1) AS repeated_rows")
    .first()["repeated_rows"]
)

print("Corrected Silver layer created successfully!")
print("Bronze rows:", bronze_loaded_df.count())
print("Silver rows:", silver_df.count())
print("Invalid records quarantined:", quarantine_df.count())
print("Repeated transaction signatures:", duplicate_signature_count)
print(
    "Fraud records preserved:",
    silver_df.filter(col("class") == 1).count()
)

Corrected Silver layer created successfully!
Bronze rows: 284807
Silver rows: 284807
Invalid records quarantined: 0
Repeated transaction signatures: 9144
Fraud records preserved: 492


In [ ]:
from pyspark.sql.functions import col, log1p

GOLD_PATH = os.path.join(
    PROJECT_PATH,
    "data/processed/gold_fraud_features"
)

# Always read the saved Silver layer
silver_loaded_df = spark.read.parquet(SILVER_PATH)

feature_columns = [
    f"v{i}" for i in range(1, 29)
]

# Create the model-ready Gold dataset
gold_df = silver_loaded_df.select(
    "transaction_id",
    *feature_columns,
    col("amount").cast("double"),
    log1p(col("amount")).alias("amount_log"),
    col("class").cast("integer").alias("is_fraud")
)

# Save by target class for efficient filtered access
(
    gold_df.write
    .mode("overwrite")
    .partitionBy("is_fraud")
    .parquet(GOLD_PATH)
)

# Read it back to verify the saved output
gold_saved_df = spark.read.parquet(GOLD_PATH)

print("Gold layer created successfully!")
print("Gold rows:", gold_saved_df.count())
print("Gold columns:", len(gold_saved_df.columns))

print("\nClass distribution:")
gold_saved_df.groupBy("is_fraud").count().orderBy("is_fraud").show()

print("Gold schema:")
gold_saved_df.printSchema()

Gold layer created successfully!
Gold rows: 284807
Gold columns: 32

Class distribution:
+--------+------+
|is_fraud| count|
+--------+------+
|       0|284315|
|       1|   492|
+--------+------+

Gold schema:
root
 |-- transaction_id: string (nullable = true)
 |-- v1: double (nullable = true)
 |-- v2: double (nullable = true)
 |-- v3: double (nullable = true)
 |-- v4: double (nullable = true)
 |-- v5: double (nullable = true)
 |-- v6: double (nullable = true)
 |-- v7: double (nullable = true)
 |-- v8: double (nullable = true)
 |-- v9: double (nullable = true)
 |-- v10: double (nullable = true)
 |-- v11: double (nullable = true)
 |-- v12: double (nullable = true)
 |-- v13: double (nullable = true)
 |-- v14: double (nullable = true)
 |-- v15: double (nullable = true)
 |-- v16: double (nullable = true)
 |-- v17: double (nullable = true)
 |-- v18: double (nullable = true)
 |-- v19: double (nullable = true)
 |-- v20: double (nullable = true)
 |-- v21: double (nullable = true)
 |-- v22: do